<a href="https://colab.research.google.com/github/kjahan/armory/blob/main/notebooks/article_summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Goal: Summarize web articles

1. Take a web page and extract its text
2. Pass text one paragraph at a time to summarizer
3. Add summaries back together

## Imports

In [4]:
import requests
from bs4 import BeautifulSoup

## Scraper

In [6]:
def scrape(url):    
    headers = {
        'authority': 'www.amazon.com',
        'pragma': 'no-cache',
        'cache-control': 'no-cache',
        'dnt': '1',
        'upgrade-insecure-requests': '1',
        'user-agent': 'Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/51.0.2704.64 Safari/537.36',
        'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
        'sec-fetch-site': 'none',
        'sec-fetch-mode': 'navigate',
        'sec-fetch-dest': 'document',
        'accept-language': 'en-GB,en-US;q=0.9,en;q=0.8',
    }

    # scrape the page using requests
    print("Scraping {}".format(url))
    r = requests.get(url, headers=headers)
    # check if page was blocked (Usually 503)
    if r.status_code > 500:
        if "To discuss automated access to Amazon data please contact" in r.text:
            print("Page {} was blocked by Amazon. Please try using better proxies".format(url))
        else:
            print("Page {} must have been blocked by Amazon as the status code was {}".format(url,r.status_code))
        return None
    # pass the HTML of the page 
    return r.text

## Input web page

Article title:
`Travel stocks fall as Omicron spurs mass flight cancellations for fourth day`

https://www.reuters.com/markets/europe/rising-omicron-cases-disrupt-air-travel-800-more-flights-canceled-2021-12-27/

In [9]:
url = "https://www.reuters.com/markets/europe/rising-omicron-cases-disrupt-air-travel-800-more-flights-canceled-2021-12-27/"

In [11]:
html = scrape(url)

Scraping https://www.reuters.com/markets/europe/rising-omicron-cases-disrupt-air-travel-800-more-flights-canceled-2021-12-27/


## Review

In [12]:
title = "Travel stocks fall as Omicron spurs mass flight cancellations for fourth day"

assert title in html

## Parse web page

In [23]:
soup = BeautifulSoup(html)

p_elems = soup.findAll('p')

# store all article paragraphs
paragraphs = []
for paragraph in p_elems:
  # print(paragraph.text)
  paragraphs.append(paragraph.text)

## Total length of all paragraphs in chars

In [55]:
original_len_chars = 0

for paragraph in paragraphs:
  original_len_chars += len(paragraph)

## Review

In [24]:
assert len(paragraphs) > 0
assert len(paragraphs[0]) > 0

## Tokenizer

In [31]:
def get_tokens(text):
  tokens = text.split()
  return tokens

## Review length of each paragraph

In [58]:
longer_paragraphs = []

max_token_threshold = 100
merged_paragraphs = []
current_tokens_no = 0

for inx in range(len(paragraphs)):
  cur_paragraph = paragraphs[inx]
  tokens = get_tokens(cur_paragraph)
  if not merged_paragraphs:
    # first paragraph to be added
    # print("Start state --> paragraph: {}".format(cur_paragraph))
    merged_paragraphs.append(cur_paragraph)
    current_tokens_no += len(tokens)
  elif current_tokens_no + len(tokens) <= max_token_threshold:
    # keep merging
    # print("keep merging --> current_tokens_no: {}".format(current_tokens_no))
    merged_paragraphs.append(cur_paragraph)
    current_tokens_no += len(tokens)
  else:
    # Done merging
    # print("Reset --> current_tokens_no: {}".format(current_tokens_no))
    new_paragraph = " ".join(merged_paragraphs)
    longer_paragraphs.append(new_paragraph)
    # Reset state
    merged_paragraphs = []
    current_tokens_no = 0


## Summarizer step

In [25]:
!pip install transformers

     |████████████████████████████████| 3.4 MB 5.2 MB/s 
     |████████████████████████████████| 61 kB 408 kB/s 
     |████████████████████████████████| 895 kB 75.1 MB/s 
     |████████████████████████████████| 596 kB 55.4 MB/s 
     |████████████████████████████████| 3.3 MB 61.5 MB/s 
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uninstalled PyYAML-3.13


## Imports

In [26]:
from transformers import pipeline

## Pipeline

In [27]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

Downloading:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.51G [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/878k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/446k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.29M [00:00<?, ?B/s]

## Process paragraph by pargraph and run the summary

In [46]:
summaries = []

for paragraph in longer_paragraphs:
    summary = summarizer(paragraph, max_length=130, min_length=30, do_sample=False)
    summaries.append(summary)

Your max_length is set to 130, but you input_length is only 110. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=55)
Your max_length is set to 130, but you input_length is only 126. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=63)


In [50]:
assert len(summaries) == len(longer_paragraphs)

## Stich back smmarized paraphs

In [53]:
pieces = []

for item in summaries:
  piece = item[0]['summary_text']
  pieces.append(piece)

overall_summary = " ".join(pieces)

## Summary Ratio

In [57]:
print("Summary ratio: {}".format(1.0*len(overall_summary)/original_len_chars))

Summary ratio: 0.27685859254356937


## Summary

In [54]:
print(overall_summary)

Over 1,000 flights were canceled within, into, or out of the United States on Monday. Globally, more than 2,600 flights were scrapped. That was on top of over 3,000 U.S. flight cancellations during the Christmas holiday weekend. Staff shortages at airlines, weather-related disruptions and now the fast-spreading Omicron variant have disrupted flights frequently this year. Most airline stocks have rallied this year on hopes of a travel boom. American Airlines says it had to cancel flights due to "COVID-related sick calls" Shanghai government suspends two China Eastern Airlines Corp Ltd flights from New York to Shanghai from Jan. 3. Carnival Corp (CCL.N) said it had isolated a small number of passengers on board its Carnival Freedom cruise ship due to positive COVID-19 test results. All passengers from the cruise trip disembarked on Sunday, and the ship departed Monday afternoon on its next planned voyage. Shares of Carnival were down 1.1%, while its peers Norwegian Cruise Line Holdings (